# Silver — Teams

Lê Delta Bronze e aplica regras de qualidade.

**Transformações:**
- Filtra nulos em `name`, `cost_center`, `owner_email`
- Deduplica por `name` (chave natural)
- Remove colunas de pipeline da Bronze

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15


In [ ]:
import sys
import os

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if __import__("os").path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

from utils.spark_session import create_spark_session, postgres_jdbc_url, postgres_props
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, StringType, DecimalType, TimestampType, DateType
)


In [ ]:
spark = create_spark_session("silver_teams")

df = spark.read.format("delta").load("s3a://datalake/bronze/teams/")

df_silver = (
    df
    .filter(F.col("name").isNotNull())
    .filter(F.col("cost_center").isNotNull())
    .filter(F.col("owner_email").isNotNull())
    .dropDuplicates(["name"])
    .drop("_ingested_at", "_source_layer")
    .withColumn("_processed_at", F.current_timestamp())
)

output_path = "s3a://datalake/silver/teams/"
df_silver.write.format("delta").mode("overwrite").save(output_path)

print(f"[silver_teams] {df_silver.count()} times → {output_path}")
df_silver.show(truncate=False)
spark.stop()
